# Step By Step 03 - 因子设计
----------

任何用户在进行因子设计的时候，都会需要进行因子的变换。**alpha-mind**依托**finance-python**完成因子的组合变换，极大的提升了因子设计的效率；|

## 一、基础因子获取

In [ ]:
import asyncio
import datetime as dt


In [ ]:
import numpy as np


In [ ]:
import pyarrow as pa
import pyarrow.compute as pc


## 二、四则运算

In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


In [ ]:
# 支持直观的加、减、乘、除
n_assets = 12


In [ ]:
# 表达式可以任意长
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


## 三、基于窗口的滚动计算

金融计算必然设计时间序列处理，而时间处理方面，必然设计滚动运算。而这方面，**finance-python**也提供了完整的支持

In [ ]:
# 一个标准的滚动平均
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


## 四、基于截面的处理

基于金融因子分析，经常需要比较个股之间的差别。这里面涉及到的就是截面的处理；比如：沪深300个股在某个指标上面的排序等：

In [ ]:
# 我们以沪深300成分股为例
from alphamind.data.transformer import Transformer


In [ ]:
from alphamind.formula import LAST, MA, SHIFT, CSQuantiles, CSRank, decode_formula, encode_formula

expressions = {
    "raw": LAST("factor"),
    "lag": SHIFT("factor", 1),
    "mean": MA(3, "factor"),
    "rank": CSRank("factor"),
    "percentile": CSQuantiles("factor"),
    "combined": LAST("factor") + 0.5 * LAST("value"),
}


## 五、全部的复合

以上所有的运算都可以相互复合

In [ ]:
result = await asyncio.to_thread(Transformer(expressions).transform, "code", panel)
restored = decode_formula(encode_formula(expressions["lag"]))


In [ ]:
restored_result = await asyncio.to_thread(Transformer({"lag": restored}).transform, "code", panel)


In [ ]:
assert result["lag"].to_pylist() == restored_result["lag"].to_pylist()
result
